# ระบบนี้ทำงานยังไง

EDR alert หนึ่งใบ เดินผ่านทุกสเตจ พร้อมแสดงผลลัพธ์ของแต่ละขั้น
กดรันจากบนลงล่างได้เลย ไม่ต้องมี Postgres และไม่ต้องมี key จริง

```
alert ดิบ
   │  Stage 0   validate_ocsf       นี่เป็น OCSF จริงหรือเปล่า
   │  Stage 1   classify_and_mark   HOST_DESKTOP, USER_PRIV, file path
   │  Stage 2   mark_cii            field rules + กวาด free text
   │  Stage 3   tokenize_log        FF3 + vault ที่บันทึก token ที่ออก
   ▼
alert ที่ปลอดภัย ──▶ LLM ──▶ คำตอบ ──▶ restore_from_llm ──▶ อ่านได้อีกครั้ง
```

จุดที่อยากให้สังเกตที่สุดอยู่ที่หัวข้อ **"ทำไม Stage 1 กับ 2 ยังไม่พอ"**
สองสเตจแรกแค่*เขียนค่าใหม่* ไม่ได้*ซ่อน*ค่า มีแต่ Stage 3 ที่ซ่อนจริง

> ledger ใน notebook นี้อยู่ในหน่วยความจำ token ทุกตัวจะหายเมื่อปิด kernel
> ใช้งานจริงต้องใช้ `connect_database()` และ key ที่สร้างเอง — ดู README

## 1. โหลด engine

In [ ]:

"""โหลด engine แล้วตั้ง key กับ ledger สำหรับสาธิต"""

import json
import os
from pathlib import Path


def locate(name):
    """หา notebook ให้เจอทั้งตอนรันใน Jupyter และตอนรันเป็นสคริปต์"""
    here = Path(__file__).parent if "__file__" in globals() else Path.cwd()
    for root in (here, here / "ver_1", here.parent, Path.cwd(), Path.cwd() / "ver_1"):
        if (root / name).exists():
            return (root / name).resolve()
    raise FileNotFoundError(name)


ENGINE = locate("encryp.ipynb")
cells = json.loads(ENGINE.read_text(encoding="utf-8"))["cells"]
exec(compile(
    "\n".join("".join(c["source"]) for c in cells
              if c["cell_type"] == "code" and "library" in c["metadata"].get("tags", [])),
    str(ENGINE), "exec",
))

# key สำหรับสาธิตเท่านั้น ของจริงสร้างเองด้วย
# secrets.token_hex(16) และ secrets.token_hex(7)
os.environ["FF3_KEY"] = "A84E9E1A54D3E108F0D1B1F69F31C1C1"
os.environ["FF3_TWEAK"] = "5777078E56B6E2"      # 14 hex = 56 บิต = FF3-1


class MemoryLedger:
    """ตัวแทนของ vault_schema.issued_tokens ไม่บันทึกลงดิสก์"""

    def __init__(self):
        self.issued, self.audit_log, self.commit_count = {}, [], 0

    def cursor(self):
        return _MemoryCursor(self)

    def commit(self):
        self.commit_count += 1


class _MemoryCursor:
    def __init__(self, ledger):
        self.ledger, self.row, self.rows, self.rowcount = ledger, None, [], -1

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        return False

    def execute(self, query, params=()):
        sql = " ".join(query.split()).upper()
        self.row, self.rows, self.rowcount = None, [], -1
        if sql.startswith("SELECT PREFIX"):
            self.row = self.ledger.issued.get(params[0])
        elif sql.startswith("SELECT KEY_VERSION"):
            counts = {}
            for _, _, version in self.ledger.issued.values():
                counts[version] = counts.get(version, 0) + 1
            self.rows = sorted(counts.items())
        elif sql.startswith("INSERT INTO VAULT_SCHEMA.ISSUED_TOKENS"):
            token, prefix, suffix_len, version = params
            if token in self.ledger.issued:
                self.rowcount = 0
            else:
                self.ledger.issued[token] = (prefix, suffix_len, version)
                self.rowcount = 1
        elif sql.startswith("INSERT INTO METADATA_SCHEMA.DETOKENIZE_LOG"):
            self.ledger.audit_log.append(params)
            self.rowcount = 1

    def fetchone(self):
        return self.row

    def fetchall(self):
        return self.rows


key_ring = load_key_ring()
vault = MemoryLedger()
print(f"โหลด engine จาก {ENGINE.name} แล้ว")
print(f"key ring: {key_ring}")

โหลด engine จาก encryp.ipynb แล้ว
key ring: KeyRing(versions=('v1',), active='v1')


## 2. ตัว alert

ใน record นี้มี 6 ค่าที่ระบุตัวบุคคลหรือโครงสร้างพื้นฐาน ไม่มีตัวไหนที่ควรหลุดถึง LLM

In [2]:

# EDR alert หนึ่งใบ ตามที่ connector ส่งมา
# บรรทัดที่มี * คือค่าที่ระบุตัวบุคคลหรือโครงสร้างพื้นฐาน ห้ามหลุดถึง LLM
ALERT = {
    "class_uid": 2004,
    "class_name": "Detection Finding",
    "severity": "High",
    "action": "Denied",
    "device": {
        "hostname": "PC-HR-02",                 # * เครื่องของใคร
        "ip": "172.20.28.67",                   # * อยู่ตรงไหนในเครือข่าย
        "type": "Desktop",
        "agent_list": [{"uid": "sangfor-agent-pc-hr-02",   # * มีชื่อเครื่องซ้ำอยู่ข้างใน
                        "name": "Sangfor Endpoint Secure Agent"}],
    },
    "metadata": {
        "log_source": "172.20.28.114 logs[6124]",          # * ตัว log collector เอง
        "product": {"vendor_name": "Sangfor", "name": "Endpoint Secure"},
    },
    "finding_info": {
        "title": "Ransomware Behavior Detection",
        "desc": "HR-User on PC-HR-02 ran a process that encrypted 24 files",
    },
    "evidences": [{
        "user": {"name": "HR-User"},            # * ใคร
        "process": {"name": "powershell.exe",
                    "parent_process": {"name": "winword.exe"}},
        "file": {"name": "update.exe",
                 "path": "C:\\Users\\Public\\Downloads\\update.exe",  # * ที่ไหน
                 "hashes": [{"algorithm": "MD5",
                             "value": "8f14e45fceea167a5a36dedd4bea2543"}]},
    }],
}

SENSITIVE = ["PC-HR-02", "172.20.28.67", "172.20.28.114", "HR-User",
             "sangfor-agent-pc-hr-02", "C:\\Users\\Public\\Downloads"]

print(json.dumps(ALERT, indent=2, ensure_ascii=False)[:520], "...")
print(f"\nมี {len(SENSITIVE)} ค่าที่ห้ามหลงเหลือใน output:")
for value in SENSITIVE:
    print(f"  - {value}")

{
  "class_uid": 2004,
  "class_name": "Detection Finding",
  "severity": "High",
  "action": "Denied",
  "device": {
    "hostname": "PC-HR-02",
    "ip": "172.20.28.67",
    "type": "Desktop",
    "agent_list": [
      {
        "uid": "sangfor-agent-pc-hr-02",
        "name": "Sangfor Endpoint Secure Agent"
      }
    ]
  },
  "metadata": {
    "log_source": "172.20.28.114 logs[6124]",
    "product": {
      "vendor_name": "Sangfor",
      "name": "Endpoint Secure"
    }
  },
  "finding_info": {
    "title": "R ...

มี 6 ค่าที่ห้ามหลงเหลือใน output:
  - PC-HR-02
  - 172.20.28.67
  - 172.20.28.114
  - HR-User
  - sangfor-agent-pc-hr-02
  - C:\Users\Public\Downloads


## 3. Stage 0 — นี่เป็น OCSF จริงหรือเปล่า

ถ้า container ผิดชนิด ตัว marker จะเดินข้ามไปเฉย ๆ แล้ว output จะดูสะอาดทั้งที่ยังมีค่าดิบอยู่ข้างใน

In [14]:

# Stage 0 — ปฏิเสธอะไรก็ตามที่ไม่ได้มีรูปร่างเป็น OCSF
# ต้องเช็คก่อนที่สเตจอื่นจะเริ่มเชื่อโครงสร้างของมัน
print("record ที่ถูกต้อง:", validate_ocsf(ALERT) is ALERT)

for broken, why in [
    ({"device": {"hostname": "PC-HR-02"}}, "ไม่มี class_uid"),
    ({"class_uid": 2004, "evidences": {"user": {}}}, "evidences เป็น dict ไม่ใช่ list"),
]:
    try:
        validate_ocsf(broken)
    except OcsfValidationError as error:
        print(f"ปฏิเสธ ({why}): {error}")

record ที่ถูกต้อง: True
ปฏิเสธ (ไม่มี class_uid): record has neither class_uid nor class_name
ปฏิเสธ (evidences เป็น dict ไม่ใช่ list): evidences should be list, got dict


## 4. Stage 1 — prefix ที่ต้องดูจากตัว record

`mark_cii` เลือก prefix จากตารางตายตัว มันพูดไม่ได้ว่า “ถ้า `device.type` เป็น Desktop ให้ใช้ HOST_DESKTOP” Stage 1 จึงต้อง mark field พวกนี้ก่อน

In [4]:

# Stage 1 — prefix ที่ต้องดูจากตัว record ไม่ใช่ดูจาก path อย่างเดียว
marked, collected = classify_and_mark(ALERT)

print("Stage 1 แทนค่าอะไรไปบ้าง:\n")
for original, placeholder in collected.items():
    print(f"  {original:36} -> {placeholder[:52]}")

print("\nประเภทเครื่องดูจาก record ก่อน แล้วค่อยเดาจากชื่อ")
print("ลองเครื่องที่ชื่อเหมือน server แต่ record บอกว่าเป็น desktop:")
print(f"  device.type='Desktop' -> "
      f"{classify_hostname('srv-legacy-box', {'type': 'Desktop'})}   <- record บอก")
print(f"  ไม่มี device.type      -> "
      f"{classify_hostname('srv-legacy-box', {})}    <- เดาจากชื่อ")
print(f"\nผู้ใช้: 'HR-User' -> {classify_username('HR-User')}, "
      f"'administrator' -> {classify_username('administrator')}")

Stage 1 แทนค่าอะไรไปบ้าง:

  PC-HR-02                             -> [HOST_DESKTOP_5438UE7KHUJOY]
  172.20.28.114 logs[6124]             -> [SRV_LOG_COLLECTOR_7247TZATKC3SSZ41BQDM31GAHDXDXZV]
  HR-User                              -> [USER_P9Y2X1U62R6]
  C:\Users\Public\Downloads\update.exe -> [FILE_PATH_USER_PUBLIC_LMCBH60GFZLUJLA6XNYZ75Z8TVPH2
  C:\Users\Public\Downloads            -> [FILE_PATH_USER_PUBLIC_LMCBH60GFZLUJLA6XNYZ75Z8TVPH2

ประเภทเครื่องดูจาก record ก่อน แล้วค่อยเดาจากชื่อ
ลองเครื่องที่ชื่อเหมือน server แต่ record บอกว่าเป็น desktop:
  device.type='Desktop' -> HOST_DESKTOP   <- record บอก
  ไม่มี device.type      -> HOST_SERVER    <- เดาจากชื่อ

ผู้ใช้: 'HR-User' -> USER, 'administrator' -> USER_PRIV


C:\apilak\Cyber-lab\open-claw-ocsf\ver_1\encryp.ipynb:1135: UserWarning: ใช้ heuristic ชั่วคราวสำหรับ desktop/server และ privileged user classification - ควรแทนที่ด้วย asset inventory / IAM lookup จริงก่อน production
  "## Stage 1 - sub-classified prefixes and file paths\n",


## 5. Stage 2 — field rules และการกวาด free text

free text เอ่ยถึงค่าเดิมซ้ำ จึงต้องได้ placeholder ตัวเดียวกัน ไม่งั้น `finding_info.desc` จะยกทุกอย่างที่ structured field เพิ่งปิดไปให้หมด

In [5]:

# Stage 2 — field rules และการกวาด free text
# สังเกต finding_info.desc: มันเอ่ยถึงค่าที่ structured field แทนไปแล้ว
# ตัวกวาดจึงใส่ placeholder ตัวเดียวกันลงไป ให้ทั้งสองที่ตรงกัน
marked = mark_cii(marked, EDR_POLICY)

print("finding_info.desc")
print(f"  ก่อน: {ALERT['finding_info']['desc']}")
print(f"  หลัง: {marked['finding_info']['desc'][:100]}")

finding_info.desc
  ก่อน: HR-User on PC-HR-02 ran a process that encrypted 24 files
  หลัง: [USER_P9Y2X1U62R6] on [HOST_DESKTOP_5438UE7KHUJOY] ran a process that encrypted 24 files


## 6. ⚠️ ทำไม Stage 1 กับ 2 ยังไม่พอ

**นี่คือขั้นที่คนมองข้ามบ่อยที่สุด** placeholder คือการเขียนค่าใหม่ ไม่ใช่การปลอมตัว

In [6]:

# Stage 1 กับ 2 ไม่ใช่ตัวปกปิด
# placeholder ยังสะกดค่าเดิมออกมาอยู่ แค่เขียนใหม่ในรูปแบบที่กลับได้
# ใครก็อ่านกลับได้โดยไม่ต้องมี key
for path in [("device", "ip"), ("device", "hostname"),
             ("evidences", 0, "user", "name")]:
    node = marked
    for key in path:
        node = node[key]
    print(f"  {'.'.join(str(k) for k in path):22} {node[:40]:42} "
          f"-> {decode_cii(node)}")

print(f"\n  ที่อยู่ IP ยังอยู่ในข้อความตรง ๆ: "
      f"{'172020028067' in json.dumps(marked)}")
print("  ถ้าส่งอันนี้ให้ LLM คือหลุดทั้งดุ้น — Stage 3 ต่างหากที่ซ่อนจริง")

  device.ip              [INT_IP_172020028067]                      -> 172.20.28.67
  device.hostname        [HOST_DESKTOP_5438UE7KHUJOY]               -> PC-HR-02
  evidences.0.user.name  [USER_P9Y2X1U62R6]                         -> HR-User

  ที่อยู่ IP ยังอยู่ในข้อความตรง ๆ: True
  ถ้าส่งอันนี้ให้ LLM คือหลุดทั้งดุ้น — Stage 3 ต่างหากที่ซ่อนจริง


## 7. Stage 3 — เข้ารหัส และ vault

ถึงตรงนี้ค่าเดิมหายไปจริง ๆ ส่วน vault บันทึกว่า token แต่ละตัว ถูกออกจริง ซึ่งเป็นสิ่งที่ทำให้จับ token ปลอมได้

In [7]:

# Stage 3 — เข้ารหัสแบบ format-preserving เฉพาะ suffix ของแต่ละ placeholder
# แล้วบันทึกทุก token ลง vault เพื่อให้ตรวจสอบย้อนหลังได้
safe = tokenize_log(marked, key_ring, vault)

leaked = [value for value in SENSITIVE if value in json.dumps(safe)]
print(f"ค่าดิบที่หลงเหลือใน output : {len(leaked)}")
print(f"token ที่ออก              : {len(vault.issued)}")
print(f"จำนวน commit             : {vault.commit_count}  (หนึ่งครั้งต่อหนึ่ง record)")
print("\nสิ่งที่ vault เก็บ — ไม่มี plaintext อยู่เลย:")
for token, (prefix, suffix_len, version) in list(vault.issued.items())[:3]:
    print(f"  {token[:44]:46} prefix={prefix:14} len={suffix_len} key={version}")

ค่าดิบที่หลงเหลือใน output : 0
token ที่ออก              : 6
จำนวน commit             : 1  (หนึ่งครั้งต่อหนึ่ง record)

สิ่งที่ vault เก็บ — ไม่มี plaintext อยู่เลย:
  [HOST_DESKTOP_B2F7Q0W4T2E9Q]                   prefix=HOST_DESKTOP   len=13 key=v1
  [INT_IP_MAZH9KQDHT5F]                          prefix=INT_IP         len=12 key=v1
  [AGENT_46T4PU9QNHQQ29ABG5GO_OPNKD4TC]          prefix=AGENT          len=29 key=v1


## 8. ทั้ง record ก่อนกับหลัง

`*` คือ field ที่เปลี่ยน ที่เหลือคือสิ่งที่ analyst และ LLM ยังต้องใช้ จึงไม่ถูกแตะ

In [8]:

# ทั้ง record เทียบก่อนกับหลัง
def walk(node, path=()):
    if isinstance(node, dict):
        for key, value in node.items():
            yield from walk(value, path + (key,))
    elif isinstance(node, list):
        for index, value in enumerate(node):
            yield from walk(value, path + (str(index),))
    else:
        yield ".".join(path), node


after = dict(walk(safe))
# หัวตารางเป็นอังกฤษเพราะวรรณยุกต์ไทยนับเป็น code point แต่ไม่กินความกว้าง
# ทำให้คอลัมน์เยื้อง
print(f"  {'field':42} {'before':30} after")
print("  " + "-" * 96)
for dotted, before in walk(ALERT):
    now = after.get(dotted)
    mark = "*" if before != now else " "
    print(f" {mark}{dotted:42} {str(before)[:28]:30} {str(now)[:40]}")

  field                                      before                         after
  ------------------------------------------------------------------------------------------------
  class_uid                                  2004                           2004
  class_name                                 Detection Finding              Detection Finding
  severity                                   High                           High
  action                                     Denied                         Denied
 *device.hostname                            PC-HR-02                       [HOST_DESKTOP_B2F7Q0W4T2E9Q]
 *device.ip                                  172.20.28.67                   [INT_IP_MAZH9KQDHT5F]
  device.type                                Desktop                        Desktop
 *device.agent_list.0.uid                    sangfor-agent-pc-hr-02         [AGENT_46T4PU9QNHQQ29ABG5GO_OPNKD4TC]
  device.agent_list.0.name                   Sangfor Endpoint Secure Agen   San

## 9. สิ่งที่ LLM เห็น

In [9]:

# นี่คือสิ่งที่ LLM จะได้รับ และตัวอย่างคำตอบที่มันอาจตอบกลับมา
# ยังไม่ได้เรียก LLM จริงในรอบนี้
print("--- สิ่งที่ส่งให้ LLM ---")
print(json.dumps(safe, ensure_ascii=False)[:400], "...\n")


def mock_model(document):
    """คำตอบสำเร็จรูปที่ใช้ token ตัวเดียวกับที่ได้รับมา"""
    device = document["device"]
    user = document["evidences"][0]["user"]["name"]
    return (
        f"Ransomware activity on {device['hostname']} ({device['ip']}), "
        f"account {user}. The process wrote to "
        f"{document['evidences'][0]['file']['path']}. Isolate the host and "
        f"reset the account."
    )


answer = mock_model(safe)
print("--- คำตอบจาก LLM ---")
print(answer)

--- สิ่งที่ส่งให้ LLM ---
{"class_uid": 2004, "class_name": "Detection Finding", "severity": "High", "action": "Denied", "device": {"hostname": "[HOST_DESKTOP_B2F7Q0W4T2E9Q]", "ip": "[INT_IP_MAZH9KQDHT5F]", "type": "Desktop", "agent_list": [{"uid": "[AGENT_46T4PU9QNHQQ29ABG5GO_OPNKD4TC]", "name": "Sangfor Endpoint Secure Agent"}]}, "metadata": {"log_source": "[SRV_LOG_COLLECTOR_5WG8583H0_TLV8O_I37O335FF_XSII0]", "product": ...

--- คำตอบจาก LLM ---
Ransomware activity on [HOST_DESKTOP_B2F7Q0W4T2E9Q] ([INT_IP_MAZH9KQDHT5F]), account [USER_2BTM80JEQJ5]. The process wrote to [FILE_PATH_USER_PUBLIC_24POUUJWY5_FFZ21J1M3HD91D6_PPFQW]\update.exe. Isolate the host and reset the account.


## 10. แปลงคำตอบกลับ

In [10]:

# คำตอบกลับมาพร้อม token ทุกตัวถูกตรวจกับ vault ก่อน
# แล้วจึงถอดกลับเป็นค่าเดิม
restored = restore_from_llm(answer, vault, key_ring, actor="analyst")

print("--- อ่านได้อีกครั้ง ---")
print(restored)
print(f"\nจำนวน audit: {len(vault.audit_log)} แถว")
for token, actor, outcome in vault.audit_log[:4]:
    print(f"  {outcome:9} {actor:9} {token[:44]}")

--- อ่านได้อีกครั้ง ---
Ransomware activity on PC-HR-02 (172.20.28.67), account HR-User. The process wrote to C:\Users\Public\Downloads\update.exe. Isolate the host and reset the account.

จำนวน audit: 4 แถว
  success   analyst   [HOST_DESKTOP_B2F7Q0W4T2E9Q]
  success   analyst   [INT_IP_MAZH9KQDHT5F]
  success   analyst   [USER_2BTM80JEQJ5]
  success   analyst   [FILE_PATH_USER_PUBLIC_24POUUJWY5_FFZ21J1M3H


## 11. token ที่ LLM แต่งขึ้นเอง

การตรวจคือ “เคยออกจริงไหม” ไม่ใช่ “ดูสมจริงไหม”

In [11]:

# ถ้า LLM แต่ง token ขึ้นมาเอง จะถูกปฏิเสธ
# ไม่ใช่ถอดออกมาเป็นที่อยู่ที่ดูสมจริง และการพยายามนั้นถูกบันทึกไว้
invented = answer.replace(safe["device"]["hostname"], "[HOST_DESKTOP_ZZZZZZZZZZZZ]")

try:
    restore_from_llm(invented, vault, key_ring, actor="analyst")
except TokenHallucinationError as error:
    print(f"ปฏิเสธ: {error}")
    print(f"บันทึกไว้: {vault.audit_log[-1]}")

ปฏิเสธ: Token was never issued: '[HOST_DESKTOP_ZZZZZZZZZZZZ]'
บันทึกไว้: ('[HOST_DESKTOP_ZZZZZZZZZZZZ]', 'analyst', 'rejected')


## 12. ค่าเดิมให้ token เดิมเสมอ

In [12]:

# ค่าเดิมให้ token เดิมเสมอ ไม่ว่าจะรันกี่ครั้งหรือ process ไหน
# นี่คือสิ่งที่ทำให้ LLM ยังโยงความสัมพันธ์ได้ทั้งที่ไม่เคยเห็นค่าจริง
second = sanitize_document(
    {"class_uid": 2004, "severity": "Low",
     "device": {"hostname": "PC-HR-02", "ip": "172.20.28.67", "type": "Desktop"}},
    key_ring, vault,
)

print(f"  record แรก  : {safe['device']['hostname']}")
print(f"  record สอง  : {second['device']['hostname']}")
print(f"  token เดียวกัน: {safe['device']['hostname'] == second['device']['hostname']}")
print(f"  vault มี     : {len(vault.issued)} token (ไม่มีตัวซ้ำเพิ่ม)")

  record แรก  : [HOST_DESKTOP_B2F7Q0W4T2E9Q]
  record สอง  : [HOST_DESKTOP_B2F7Q0W4T2E9Q]
  token เดียวกัน: True
  vault มี     : 6 token (ไม่มีตัวซ้ำเพิ่ม)


## 13. ทำแบบเดียวกันกับทั้งไฟล์

In [13]:

# ทำแบบเดียวกันกับทั้งไฟล์ พร้อมสแกนสิ่งที่เขียนออกไปจริง ๆ
output_path, raw_values = run_pipeline(key_ring=key_ring, conn=vault, limit=500)

print()
export_tokens_csv(limit=500)

Stages 1-3 complete: 500 records -> C:\apilak\Cyber-lab\open-claw-ocsf\Data\ocsf_sanitized.log
  distinct tokens issued : 24
  file_directory     1 distinct raw value(s), 0 in output
  hostname           5 distinct raw value(s), 0 in output
  internal_ip        6 distinct raw value(s), 0 in output
  username           6 distinct raw value(s), 0 in output

24 rows -> C:\apilak\Cyber-lab\open-claw-ocsf\Data\ocsf_tokens.csv
  AGENT                       5
  HOST_DESKTOP                5
  INT_IP                      5
  USER                        5
  EXT_IP                      1
  FILE_PATH_USER_PUBLIC       1
  SRV_LOG_COLLECTOR           1
  USER_PRIV                   1


WindowsPath('C:/apilak/Cyber-lab/open-claw-ocsf/Data/ocsf_tokens.csv')

## สิ่งที่ notebook นี้ยังไม่ได้ทำ

- **ยังไม่ได้เรียก LLM จริง** `mock_model` เป็นสตริงสำเร็จรูป
  การต่อกับ LLM จริงเป็นงานรอบถัดไป
- **ยังไม่มี validator** ที่ตรวจว่าคำตอบใช้เฉพาะ token ที่ส่งไปให้จริง ๆ
- **ยังไม่มี anti-prompt-injection layer** เอกสารที่ผ่านการ sanitize แล้ว
  ยังพา free text ที่ผู้โจมตีควบคุมได้เข้าไปใน prompt อยู่ดี
  รอบนี้จึงใช้ได้กับข้อมูลสังเคราะห์เท่านั้น
- **ledger อยู่ในหน่วยความจำ** ไม่มีอะไรเหลือหลังปิด kernel

ตัวโค้ดจริงและเทสต์ 205 ตัวอยู่ใน `encryp.ipynb` กับ `test_encryp.ipynb`